In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, TimeSeriesSplit
import joblib
import json
from pathlib import Path

from DataTransformationPipeline import RowDataPreprocessingPipeline
from ModelTrainer import ModelTrainerForEachMachineType

In [2]:
bundle = joblib.load(r"C:\Users\devan\Desktop\SAR_Work\forecasting_ready_bundle.pkl")
models = bundle["models"]
features = bundle["feature_columns"]
final_df_from_bundle = bundle["all_machine_type_data"]

In [3]:
data_transformation = RowDataPreprocessingPipeline()
DATA_PATH = r"C:\Users\devan\Desktop\SAR_Work\Data\Original_Data\EnergyParameterDataset.xlsx"
final_df = data_transformation.load_and_transform_data(DATA_PATH)

In [4]:
final_df.columns

Index(['HOURLY_KWH', 'AVG_CURRENT', 'AVG_V_LN', 'original_data', 'power_proxy',
       'hour', 'weekday', 'month', 'week_of_year', 'hour_sin', 'hour_cos',
       'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'Shift_A',
       'Shift_B', 'Shift_C', 'Off', 'kwh_lag_1', 'kwh_lag_2', 'kwh_lag_24',
       'kwh_lag_168', 'kwh_roll_3h_mean', 'kwh_roll_24h_mean',
       'kwh_roll_24h_std', 'kwh_roll_24h_min', 'kwh_roll_24h_max',
       'kwh_roll_168h_mean', 'kwh_roll_168h_std', 'kwh_ratio_to_24h_avg',
       'kwh_ratio_to_168h_avg', 'Type'],
      dtype='object')

In [5]:
df_YWNC2_CONE = final_df[final_df['Type'].isin(['YWNC2 CONE'])].drop(columns="Type")
df_YWNC2_CUP  = final_df[final_df['Type'].isin(['YWNC2 CUP'])].drop(columns="Type")
df_YWNC3_CONE = final_df[final_df['Type'].isin(['YWNC3 CONE'])].drop(columns="Type")
df_YWNC3_CUP  = final_df[final_df['Type'].isin(['YWNC3 CUP'])].drop(columns="Type")

In [6]:
df_YWNC2_CONE[df_YWNC2_CONE['HOURLY_KWH'] >= 1].shape

(3355, 32)

In [7]:
def plot_weekly_mean(df, title):
    weekly_mean = df['HOURLY_KWH'].resample('W-MON').mean()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=weekly_mean.index,
        y=weekly_mean.values,
        mode='lines+markers',
        name='Weekly Mean HOURLY_KWH',
        hovertemplate="<b>Date:</b> %{x}<br><b>Weekly Mean:</b> %{y:.2f}<extra></extra>"
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Week",
        yaxis_title="Weekly Mean HOURLY_KWH",
        template="plotly_white",
        hovermode="x unified",
        xaxis=dict(
            rangeslider=dict(visible=True),
            type="date"
        )
    )

    fig.show()

plot_weekly_mean(df_YWNC2_CONE, "YWNC2 CONE – Weekly Mean")
plot_weekly_mean(df_YWNC2_CUP,  "YWNC2 CUP – Weekly Mean")
plot_weekly_mean(df_YWNC3_CONE, "YWNC3 CONE – Weekly Mean")
plot_weekly_mean(df_YWNC3_CUP,  "YWNC3 CUP – Weekly Mean")

The total of all values over a 7-day period is calculated, and the result is displayed on the 7th day of that period.


In [10]:
json_path = Path(r"C:\Users\devan\Desktop\SAR_Work\1st_to_6th_feb.json")

with open(json_path, "r") as f:
    json_data = json.load(f)

json_df = pd.DataFrame(json_data)

In [11]:
json_df.head()

,Time,TOTAL_NET_KWH,AVG_CURRENT,AVG_V_LL,AVG_V_LN,FREQUENCY,Type
0,"01/02/2026, 24:00:43",12829.169,5.620,411.750,237.814,50.025,YWNC2 CONE
1,"01/02/2026, 24:03:20",12829.237,5.656,411.695,237.668,50.010,YWNC2 CONE
2,"01/02/2026, 24:05:53",12829.303,6.978,411.118,237.367,49.979,YWNC2 CONE
3,"01/02/2026, 24:08:36",12829.370,4.417,411.202,237.273,49.967,YWNC2 CONE
4,"01/02/2026, 24:11:16",12829.420,4.404,410.133,236.725,49.921,YWNC2 CONE


In [12]:
json_df['Type'].unique()

array(['YWNC2 CONE', 'YWNC2 CUP', 'YWNC3 CONE', 'YWNC3 CUP'], dtype=object)